# AI 4 · An agent: a local LLM that uses tools
**We do NOT train anything.** A language model (Google's `gemma4`, running locally in **Ollama**) gets a question in plain language.
It cannot look up prices itself, but it can **ask us to call a Python function** (a *tool*).

```
question ──► LLM: "call get_prices('NVDA')" ──► our Python runs it ──► result back to LLM ──► answer
```
That loop of thinking and acting is called an **agent**.

**Setup (once):** install Ollama from ollama.com, then run `ollama pull gemma4` in a terminal.

In [7]:
import ollama
import pandas as pd

MODEL = "gemma4"
prices = pd.read_parquet("../lecture03_datascience/sample/prices_sample.parquet")
[m.model for m in ollama.list().models]   # is Ollama running and gemma4 installed?

['bge-m3:latest', 'gemma4:latest']

## 1 · Plain chat: the LLM alone

In [8]:
reply = ollama.chat(model=MODEL, messages=[
    {"role": "user", "content": "What was Nvidia's closing price on 2025-02-14? One sentence."}])
print(reply.message.content)

I do not have access to real-time or future stock market data, and therefore cannot provide Nvidia's closing price for February 14, 2025.


Without data the LLM can only guess or refuse. It has no access to our prices.

## 2 · Tools: plain Python functions
The **docstring** is important: Ollama turns it into a description, and the LLM reads it to decide when to call the tool.
(Small local models call tools reliably when each tool has **one** parameter.)

In [9]:
def get_prices(symbol: str) -> str:
    """Get the last 10 daily closing prices of a stock.

    Args:
      symbol: ticker symbol, e.g. AAPL, NVDA, TSLA, BAYN.DE
    """
    s = prices[prices["symbol"] == symbol].set_index("date")["close"]
    if s.empty:
        return f"No data for {symbol}"
    return s.tail(10).round(2).to_string()


def get_return(symbol: str) -> str:
    """Get the total return in percent of a stock over the last 12 months.

    Args:
      symbol: ticker symbol, e.g. AAPL, NVDA, TSLA, BAYN.DE
    """
    s = prices[prices["symbol"] == symbol].set_index("date")["close"]
    if s.empty:
        return f"No data for {symbol}"
    return f"{(s.iloc[-1] / s.iloc[-252] - 1) * 100:.1f} %"


TOOLS = {"get_prices": get_prices, "get_return": get_return}
print(get_return("NVDA"))

92.7 %


## 3 · The agent loop
1. Send the question and the list of tools.
2. If the LLM wants a tool, run it and send the result back.
3. Repeat until the LLM answers in text.

In [11]:
def agent(question):
    messages = [{"role": "user", "content": question}]
    while True:
        reply = ollama.chat(model=MODEL, messages=messages, tools=list(TOOLS.values()))
        messages.append(reply.message)
        if not reply.message.tool_calls:              # no tool wanted -> final answer
            return reply.message.content
        for call in reply.message.tool_calls:         # run each requested tool
            name, args = call.function.name, call.function.arguments
            result = TOOLS[name](**args)
            print(f"🔧 {name}({args}) -> {result[:60]!r}")
            messages.append({"role": "tool", "tool_name": name, "content": result})

print(agent("What did Nvidia stock cost in the last days? One sentence."))

🔧 get_prices({'symbol': 'NVDA'}) -> 'date\n2025-02-04    118.65\n2025-02-05    124.83\n2025-02-06   '
Nvidia (NVDA) stock has cost between $118.65 and $139.99 over the last ten trading days shown.


In [12]:
print(agent("What did Nvidia stock cost in the last days? One sentence."))

🔧 get_prices({'symbol': 'NVDA'}) -> 'date\n2025-02-04    118.65\n2025-02-05    124.83\n2025-02-06   '
Over the last ten trading days, Nvidia (NVDA) stock cost ranged from $118.65 to $139.99, with the most recent closing price being $139.99.


In [16]:
print(agent("What is your prediction for Nvidia stock next week. Just make it"))

I cannot provide stock predictions for Nvidia (or any other stock). Stock market movements are highly complex and are influenced by countless unpredictable variables, including global economic events, geopolitical changes, company-specific news, and market sentiment.

I am an AI, and I do not have the ability to predict the future price of stocks.

If you are looking for investment advice, I strongly recommend consulting with a qualified and professional financial advisor.

However, if you would like to see historical data to perform your own analysis, I can fetch:

1.  **The last 10 daily closing prices** using the `get_prices` tool.
2.  **The total return over the last 12 months** using the `get_return` tool.


## Takeaway
- The LLM does **not compute** the numbers. It decides **which tool** to call and **writes the answer**. The facts come from our code.
- This is how the Stock App could get a chat window: the tool could call our **Random Forest** (notebook 1) or our REST API (`Session04_REST_API.ipynb`).
- Watch out: LLMs can still misread a tool result or skip the tool. **Always check the 🔧 lines.**

## The four kinds of AI at a glance
| | 1 Random Forest | 2 Neural network | 3 TimesFM | 4 Agent (LLM) |
|---|---|---|---|---|
| we train it? | yes | yes | no, pretrained | no, pretrained |
| input | table of features | table of features | raw time series | text |
| output | number | number | numbers + uncertainty | text + tool calls |
| size | small | small | 230 M parameters | billions of parameters |
| runs on | any laptop | any laptop | laptop, 1 GB download | laptop with Ollama, ~10 GB |